In [1]:
import os
import sys
import json
import time
import kaggle
from kagglehub.competition import competition_download

import numpy as np
import pandas as pd

kaggle_config_dir = os.environ.get("KAGGLE_CONFIG_DIR", os.path.expanduser("~/.config/kaggle"))
with open(os.path.join(kaggle_config_dir, "kaggle.json")) as f:
    _kaggle_creds = json.load(f)
os.environ.setdefault("KAGGLE_USERNAME", _kaggle_creds["username"])
os.environ.setdefault("KAGGLE_KEY", _kaggle_creds["key"])
path = competition_download('ing-hubs-turkiye-datathon')

/home/osman-tekdamar/Projects/forCV/ING_Datathon/.claude/worktrees/init-claude-md/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
customer_history = pd.read_csv(f"{path}/customer_history.csv")
customers = pd.read_csv(f"{path}/customers.csv")
referance_data = pd.read_csv(f"{path}/referance_data.csv")
referance_data_test = pd.read_csv(f"{path}/referance_data_test.csv")
sample_submission = pd.read_csv(f"{path}/sample_submission.csv") 

In [3]:
customers["work_sector"] = customers["work_sector"].fillna(customers["work_type"])

In [4]:
customers["cust_age_month"] = (customers["age"] * 12)  - customers["tenure"]

In [5]:
def _ols_slope(y, x):
    """Basit doğrusal regresyon eğimi; n > 1 için np.polyfit(x, y, 1)[0] ile matematiksel olarak eşdeğerdir."""
    mean_x = x.mean()
    mean_y = y.mean()
    cov = ((x - mean_x) * (y - mean_y)).sum()
    var = ((x - mean_x) ** 2).sum()
    return cov / var


def create_customer_features(df):
    """
    customer_history tablosundan cust_id bazlı özellikler üretir (vektörize).

    - Kanal bazlı eksiklikler (nan) müşteriye özgü → -2 ile doldurulur (kullanım yok).
    - Kullanım var/yok bilgisi (binary flag) eklenir.
    - Trend sütunları np.polyfit(range(n), y, 1)[0] ile matematiksel olarak eşdeğer
      kapalı-form OLS eğim formülüyle hesaplanır.

    Girdi:
        df (pd.DataFrame): Verilen sütunları içeren DataFrame

    Çıktı:
        features_df (pd.DataFrame): Her müşteri için türetilmiş özellikler
    """
    df = df.copy()
    df['date'] = pd.to_datetime(df['date'])

    g = df.groupby('cust_id', sort=False)

    mobile_all_nan = g['mobile_eft_all_cnt'].transform(lambda s: s.isna().all())
    cc_all_nan = g['cc_transaction_all_cnt'].transform(lambda s: s.isna().all())

    df.loc[mobile_all_nan, 'mobile_eft_all_cnt'] = -2
    df.loc[mobile_all_nan, 'mobile_eft_all_amt'] = -2
    df.loc[cc_all_nan, 'cc_transaction_all_cnt'] = -2
    df.loc[cc_all_nan, 'cc_transaction_all_amt'] = -2

    def _fill_product(s):
        if s.isna().all():
            return pd.Series(0, index=s.index)
        return s.ffill().bfill().fillna(0)

    df['active_product_category_nbr'] = g['active_product_category_nbr'].transform(_fill_product)

    # her müşteri içinde tarihe göre sırala (orijinal koddaki group.sort_values('date') ile eşdeğer)
    df = df.sort_values(['cust_id', 'date'], kind='mergesort')
    g = df.groupby('cust_id', sort=False)

    uses_mobile_eft = (~mobile_all_nan).groupby(df['cust_id']).first()
    uses_cc = (~cc_all_nan).groupby(df['cust_id']).first()

    agg = g.agg({
        'mobile_eft_all_cnt': ['mean', 'std', 'min', 'max', 'last', 'sum'],
        'mobile_eft_all_amt': ['mean', 'std', 'min', 'max', 'last', 'sum'],
        'cc_transaction_all_cnt': ['mean', 'std', 'min', 'max', 'last', 'sum'],
        'cc_transaction_all_amt': ['mean', 'std', 'min', 'max', 'last', 'sum'],
        'active_product_category_nbr': ['mean', 'std', 'min', 'max', 'last'],
        'date': ['min', 'max', 'size'],
    })
    agg.columns = ['_'.join(c) for c in agg.columns]

    sizes = agg['date_size']
    single_row_mask = sizes <= 1
    # tek gözlemli gruplarda pandas .std() NaN döner; orijinal kod bu durumda 0 kullanıyordu
    std_cols = [
        'mobile_eft_all_cnt_std', 'mobile_eft_all_amt_std',
        'cc_transaction_all_cnt_std', 'cc_transaction_all_amt_std',
        'active_product_category_nbr_std',
    ]
    agg.loc[single_row_mask, std_cols] = 0

    def _trend(s):
        n = len(s)
        if n <= 1:
            return 0.0
        x = np.arange(n, dtype=float)
        return _ols_slope(s.to_numpy(dtype=float), x)

    def _change_1m(s):
        return (s.iloc[-1] - s.iloc[-2]) if len(s) >= 2 else 0

    result = pd.DataFrame(index=agg.index)
    result['cust_id'] = agg.index
    result['uses_mobile_eft'] = uses_mobile_eft.astype(int).reindex(agg.index)
    result['uses_cc'] = uses_cc.astype(int).reindex(agg.index)
    result['uses_any_digital_channel'] = ((result['uses_mobile_eft'] == 1) | (result['uses_cc'] == 1)).astype(int)

    result['mobile_eft_cnt_mean'] = agg['mobile_eft_all_cnt_mean']
    result['mobile_eft_cnt_std'] = agg['mobile_eft_all_cnt_std']
    result['mobile_eft_cnt_min'] = agg['mobile_eft_all_cnt_min']
    result['mobile_eft_cnt_max'] = agg['mobile_eft_all_cnt_max']
    result['mobile_eft_cnt_last'] = agg['mobile_eft_all_cnt_last']
    result['mobile_eft_cnt_trend'] = g['mobile_eft_all_cnt'].apply(_trend).reindex(agg.index)

    result['mobile_eft_amt_mean'] = agg['mobile_eft_all_amt_mean']
    result['mobile_eft_amt_std'] = agg['mobile_eft_all_amt_std']
    result['mobile_eft_amt_min'] = agg['mobile_eft_all_amt_min']
    result['mobile_eft_amt_max'] = agg['mobile_eft_all_amt_max']
    result['mobile_eft_amt_last'] = agg['mobile_eft_all_amt_last']
    result['mobile_eft_amt_trend'] = g['mobile_eft_all_amt'].apply(_trend).reindex(agg.index)

    mobile_cnt_sum = agg['mobile_eft_all_cnt_sum']
    mobile_amt_sum = agg['mobile_eft_all_amt_sum']
    result['mobile_eft_avg_amt_per_tx'] = np.where(mobile_cnt_sum > 0, mobile_amt_sum / mobile_cnt_sum, 0)

    result['cc_cnt_mean'] = agg['cc_transaction_all_cnt_mean']
    result['cc_cnt_std'] = agg['cc_transaction_all_cnt_std']
    result['cc_cnt_min'] = agg['cc_transaction_all_cnt_min']
    result['cc_cnt_max'] = agg['cc_transaction_all_cnt_max']
    result['cc_cnt_last'] = agg['cc_transaction_all_cnt_last']
    result['cc_cnt_trend'] = g['cc_transaction_all_cnt'].apply(_trend).reindex(agg.index)

    result['cc_amt_mean'] = agg['cc_transaction_all_amt_mean']
    result['cc_amt_std'] = agg['cc_transaction_all_amt_std']
    result['cc_amt_min'] = agg['cc_transaction_all_amt_min']
    result['cc_amt_max'] = agg['cc_transaction_all_amt_max']
    result['cc_amt_last'] = agg['cc_transaction_all_amt_last']
    result['cc_amt_trend'] = g['cc_transaction_all_amt'].apply(_trend).reindex(agg.index)

    cc_cnt_sum = agg['cc_transaction_all_cnt_sum']
    cc_amt_sum = agg['cc_transaction_all_amt_sum']
    result['cc_avg_amt_per_tx'] = np.where(cc_cnt_sum > 0, cc_amt_sum / cc_cnt_sum, 0)

    result['active_product_mean'] = agg['active_product_category_nbr_mean']
    result['active_product_std'] = agg['active_product_category_nbr_std']
    result['active_product_min'] = agg['active_product_category_nbr_min']
    result['active_product_max'] = agg['active_product_category_nbr_max']
    result['active_product_last'] = agg['active_product_category_nbr_last']
    result['active_product_trend'] = g['active_product_category_nbr'].apply(_trend).reindex(agg.index)
    result['active_product_recent_3m_mean'] = g['active_product_category_nbr'].apply(
        lambda s: s.tail(3).mean() if len(s) >= 3 else s.mean()
    ).reindex(agg.index)

    result['tenure_months'] = sizes
    result['first_activity_month'] = agg['date_min']
    result['last_activity_month'] = agg['date_max']

    result['cc_amt_change_1m'] = g['cc_transaction_all_amt'].apply(_change_1m).reindex(agg.index)
    result['mobile_eft_amt_change_1m'] = g['mobile_eft_all_amt'].apply(_change_1m).reindex(agg.index)
    result['active_product_change_1m'] = g['active_product_category_nbr'].apply(_change_1m).reindex(agg.index)

    result['mobile_eft_active_recent_2m'] = g['mobile_eft_all_cnt'].apply(
        lambda s: s.tail(2).sum() > 0
    ).reindex(agg.index).astype(int)
    result['cc_active_recent_2m'] = g['cc_transaction_all_cnt'].apply(
        lambda s: s.tail(2).sum() > 0
    ).reindex(agg.index).astype(int)

    result = result.reset_index(drop=True)
    result = result.fillna(-1)

    return result

In [6]:
referance_data["ref_date"] = pd.to_datetime(referance_data["ref_date"])
referance_data_test["ref_date"] = pd.to_datetime(referance_data_test["ref_date"])
customer_history["date"] = pd.to_datetime(customer_history["date"])

def filter_history_before_ref_date(history_df, reference_df):
    """
    Her müşteri için, o müşterinin ref_date'inden SONRAKİ ay kayıtlarını eler.
    Bu filtre olmadan create_customer_features, churn kararının verildiği
    tarihten sonraki (henüz gerçekleşmemiş) işlem verisini kullanıyordu.
    """
    merged = history_df.merge(reference_df[["cust_id", "ref_date"]], on="cust_id", how="inner")
    return merged[merged["date"] <= merged["ref_date"]].drop(columns=["ref_date"])

train_history = filter_history_before_ref_date(customer_history, referance_data)
test_history = filter_history_before_ref_date(customer_history, referance_data_test)

In [7]:
history_statics_train = create_customer_features(train_history)
history_statics_test = create_customer_features(test_history)

In [8]:
new_customers_train = customers.merge(history_statics_train, "left", on="cust_id")
new_customers_test = customers.merge(history_statics_test, "left", on="cust_id")

train_data = new_customers_train.merge(referance_data, "right", on="cust_id")
test_data = new_customers_test.merge(referance_data_test, "right", on="cust_id")

In [9]:
cat_cols = [col for col in train_data.columns if train_data[col].dtype == "object"]

In [10]:
# Yüksek kardinaliteli kategorik sütunlarda seyrek kategorileri "Other" altında
# toplayarak get_dummies'in ürettiği sütun sayısının şişmesini ve train/test
# arasında farklı kategori kümelerinden doğan sütun uyumsuzluğunu önlüyoruz.
# Eşik SADECE train dağılımından hesaplanıyor (test'e sızıntı yok); kategori
# kümesi de train'e göre sabitleniyor ki get_dummies her iki sette de aynı
# sütunları üretsin ("Other" her zaman kümede yer alır — test'te train'de hiç
# görülmemiş bir kategori çıkarsa da bu şekilde NaN'a düşmez).
RARE_CATEGORY_THRESHOLD = 0.01  # train setinin %1'inden azına sahip kategoriler "Other" olur

for col in cat_cols:
    value_counts = train_data[col].value_counts(normalize=True)
    frequent_categories = value_counts[value_counts >= RARE_CATEGORY_THRESHOLD].index

    train_rare_mask = train_data[col].notna() & ~train_data[col].isin(frequent_categories)
    test_rare_mask = test_data[col].notna() & ~test_data[col].isin(frequent_categories)
    train_data.loc[train_rare_mask, col] = "Other"
    test_data.loc[test_rare_mask, col] = "Other"

    categories = sorted(set(frequent_categories) | {"Other"})
    train_data[col] = pd.Categorical(train_data[col], categories=categories)
    test_data[col] = pd.Categorical(test_data[col], categories=categories)

In [11]:
ohe_train_df = pd.get_dummies(train_data[cat_cols], drop_first=True).astype(int)
ohe_test_df = pd.get_dummies(test_data[cat_cols], drop_first=True).astype(int)

In [12]:
new_train = pd.concat([train_data, ohe_train_df], axis=1).drop(cat_cols, axis=1)
new_test = pd.concat([test_data, ohe_test_df], axis=1).drop(cat_cols, axis=1)

In [13]:
new_train = new_train.drop(["cust_id", "last_activity_month", "first_activity_month","ref_date"], axis=1)
new_test = new_test.drop(["cust_id", "last_activity_month", "first_activity_month","ref_date"], axis=1)

In [14]:
new_train.to_csv("train_data.csv",index=False)
new_test.to_csv("test_data.csv", index=False)
sample_submission.to_csv("sample_submission.csv", index=False)